## Elección del modelo de clasificación🔬 

En este notebook testeo posibles algoritmos de clasificación una vez obtenidas las caras vectorizadas y con sus respectivos labels de emoción.

Pruebo distintos modelos...

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier

In [2]:
def _load_faces(dir: str) -> pd.DataFrame:    
    happy_faces = pd.read_csv(f"{dir}/happy_faces.csv", header=None).values
    sad_faces = pd.read_csv(f"{dir}/sad_faces.csv", header=None).values
    neutral_faces = pd.read_csv(f"{dir}/neutral_faces.csv", header=None).values
    # angry_faces = pd.read_csv(f"{dir}/angry_faces.csv", header=None).values
    # surprised_faces = pd.read_csv(f"{dir}/surprise_faces.csv", header=None).values
    # disgusted_faces = pd.read_csv(f"{dir}/disgust_faces.csv", header=None).values
    
    #concat every  face into a single matrix with its corresponding label
    _faces = np.concatenate([
        happy_faces,
        sad_faces,
        neutral_faces,
        # angry_faces,
        # surprised_faces,    
        # disgusted_faces
    ], axis=0)

    #labels for each face
    labels = np.concatenate([
        np.full(happy_faces.shape[0], "happy"),
        np.full(sad_faces.shape[0], "sad"),
        np.full(neutral_faces.shape[0], "neutral"),
        # np.full(angry_faces.shape[0], "angry"),
        # np.full(surprised_faces.shape[0], "surprised"),
        # np.full(disgusted_faces.shape[0], "disgusted")
    ], axis=0)

    faces_df = pd.DataFrame(_faces)

    # if last column is not label, add it
    if faces_df[faces_df.columns[-1]].dtype != object:
        faces_df['label'] = labels
    else:
        faces_df.rename(columns={faces_df.columns[-1]: 'label'}, inplace=True)

    return faces_df

pca_faces = _load_faces("../data/processed/pca_faces")

In [3]:
print(pca_faces.groupby('label').size().reset_index(name='count').to_markdown(index=False))

| label   |   count |
|:--------|--------:|
| happy   |    3884 |
| neutral |    2775 |
| sad     |    1545 |


### Función para visualizar matrices de confusión

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name, class_names=None):
    """
    Visualiza la matriz de confusión para un modelo dado.
    
    Parameters:
    -----------
    y_true : array-like
        Etiquetas verdaderas
    y_pred : array-like
        Predicciones del modelo
    model_name : str
        Nombre del modelo para el título
    class_names : array-like, optional
        Nombres de las clases. Si es None, se usan las clases únicas de y_true
    """
    cm = confusion_matrix(y_true, y_pred)
    
    if class_names is None:
        class_names = sorted(np.unique(np.concatenate([y_true, y_pred])))
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, 
                yticklabels=class_names,
                cbar_kws={'label': 'Cantidad'})
    plt.title(f'Matriz de Confusión - {model_name}', fontsize=16, fontweight='bold')
    plt.ylabel('Etiqueta Real', fontsize=12)
    plt.xlabel('Etiqueta Predicha', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Imprimir métricas adicionales
    print(f"\n{'='*60}")
    print(f"Reporte de Clasificación - {model_name}")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, target_names=class_names))
    print(f"{'='*60}\n")

### Tests con datos del IVFClassifier C++

Carga exactamente los mismos archivos que `IVFClassifier::loadIVFIndex()`:
- `data/ivf/vectors.csv` - vectores PCA ordenados por cluster
- `data/ivf/labels.csv` - índices de emociones (0-6)

Esto garantiza resultados 100% comparables con el clasificador C++.

In [ ]:
# Cargar EXACTAMENTE los mismos datos que usa IVFClassifier en C++
# Archivos: data/ivf/vectors.csv y data/ivf/labels.csv
# Ya están filtrados a las 3 emociones: happy, neutral, sad

print("Cargando datos de ../data/ivf/ (mismos que IVFClassifier)...")

# Las 3 emociones que usa C++ (definidas en types.h)
# El índice IVF fue construido con estas 3 emociones, por lo que labels ya son 0,1,2
EMOTION_CATEGORIES = ['happy', 'neutral', 'sad']

# Vectores [N x D] - ordenados por cluster
vectors_cpp = pd.read_csv("../data/ivf/vectors.csv", header=None).values
print(f"  vectors.csv: {vectors_cpp.shape}")

# Labels [N] - índices numéricos: happy=0, neutral=1, sad=2
labels_raw = pd.read_csv("../data/ivf/labels.csv", header=None).values.flatten()
labels_cpp = np.array([EMOTION_CATEGORIES[i] for i in labels_raw])
print(f"  labels.csv: {len(labels_raw)} labels")

print(f"\nTotal: {len(vectors_cpp)} vectores")
print(f"Dimensión: {vectors_cpp.shape[1]}")
print(f"\nDistribución:")
print(pd.Series(labels_cpp).value_counts())

# Split para test
X_train_cpp, X_test_cpp, y_train_cpp, y_test_cpp = train_test_split(
    vectors_cpp, labels_cpp, test_size=0.2, random_state=42, stratify=labels_cpp
)
print(f"\nTrain: {len(X_train_cpp)}, Test: {len(X_test_cpp)}")

In [ ]:
# Configuraciones a testear (mismas que en classifier.cpp)
configs = [
    # (k, metric, weights, nombre)
    (3, 'euclidean', 'uniform', 'KNN k=3 Euclidean'),
    (5, 'euclidean', 'uniform', 'KNN k=5 Euclidean'),
    (11, 'euclidean', 'uniform', 'KNN k=11 Euclidean'),
    (3, 'cosine', 'uniform', 'KNN k=3 Cosine'),
    (5, 'cosine', 'uniform', 'KNN k=5 Cosine'),
    (11, 'cosine', 'uniform', 'KNN k=11 Cosine'),
    (3, 'euclidean', 'distance', 'KNN k=3 Euclidean (weighted)'),
    (5, 'euclidean', 'distance', 'KNN k=5 Euclidean (weighted)'),
    (11, 'euclidean', 'distance', 'KNN k=11 Euclidean (weighted)'),
    (3, 'cosine', 'distance', 'KNN k=3 Cosine (weighted)'),
    (5, 'cosine', 'distance', 'KNN k=5 Cosine (weighted)'),
    (11, 'cosine', 'distance', 'KNN k=11 Cosine (weighted)'),
]

results_cpp = []

print("Testeando configuraciones con datos C++...\n")
for k, metric, weights, name in configs:
    knn = KNeighborsClassifier(n_neighbors=k, metric=metric, weights=weights)
    knn.fit(X_train_cpp, y_train_cpp)
    accuracy = knn.score(X_test_cpp, y_test_cpp)
    results_cpp.append({
        'Modelo': name,
        'k': k,
        'Métrica': metric,
        'Pesos': weights,
        'Accuracy': accuracy
    })
    print(f"{name}: {accuracy:.4f}")

In [ ]:
# Tabla comparativa
results_df = pd.DataFrame(results_cpp).sort_values('Accuracy', ascending=False)
print("\n" + "="*60)
print("RANKING DE CONFIGURACIONES (datos C++)")
print("="*60)
display(results_df)

# Gráfico comparativo
plt.figure(figsize=(14, 6))
colors = ['#2ecc71' if 'Cosine' in m else '#3498db' for m in results_df['Modelo']]
bars = plt.barh(results_df['Modelo'], results_df['Accuracy'], color=colors)
plt.xlabel('Accuracy', fontsize=12)
plt.title('Comparación de Hiperparámetros KNN (datos pipeline C++)', fontsize=14, fontweight='bold')
plt.xlim(0, max(results_df['Accuracy']) + 0.05)

for bar, acc in zip(bars, results_df['Accuracy']):
    plt.text(acc + 0.005, bar.get_y() + bar.get_height()/2, 
             f'{acc:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

# Mejor configuración
best = results_df.iloc[0]
print(f"\n>> MEJOR CONFIG: {best['Modelo']} (Accuracy: {best['Accuracy']:.4f})")

In [ ]:
# Matriz de confusión del mejor modelo
best_config = results_df.iloc[0]
print(f"Mejor configuración: {best_config['Modelo']} (Accuracy: {best_config['Accuracy']:.4f})")

best_knn = KNeighborsClassifier(
    n_neighbors=int(best_config['k']), 
    metric=best_config['Métrica'], 
    weights=best_config['Pesos']
)
best_knn.fit(X_train_cpp, y_train_cpp)
y_pred_best = best_knn.predict(X_test_cpp)

plot_confusion_matrix(y_test_cpp, y_pred_best, f"{best_config['Modelo']} (datos C++)")

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Comparar Euclidean vs Cosine (k=11, weighted) por clase
# Solo las 3 emociones: happy, neutral, sad

knn_euclidean = KNeighborsClassifier(n_neighbors=11, metric='euclidean', weights='distance')
knn_cosine = KNeighborsClassifier(n_neighbors=11, metric='cosine', weights='distance')

knn_euclidean.fit(X_train_cpp, y_train_cpp)
knn_cosine.fit(X_train_cpp, y_train_cpp)

y_pred_euc = knn_euclidean.predict(X_test_cpp)
y_pred_cos = knn_cosine.predict(X_test_cpp)

# F1-score por clase (solo las 3 emociones)
prec_euc, rec_euc, f1_euc, _ = precision_recall_fscore_support(y_test_cpp, y_pred_euc, labels=EMOTION_CATEGORIES)
prec_cos, rec_cos, f1_cos, _ = precision_recall_fscore_support(y_test_cpp, y_pred_cos, labels=EMOTION_CATEGORIES)

comparison_df = pd.DataFrame({
    'Emoción': EMOTION_CATEGORIES,
    'F1 Euclidean': f1_euc,
    'F1 Cosine': f1_cos,
    'Diferencia': f1_cos - f1_euc,
    'Mejor': ['Cosine' if d > 0 else 'Euclidean' for d in (f1_cos - f1_euc)]
})

print("Comparación F1-Score por emoción (3 clases):")
display(comparison_df)

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(EMOTION_CATEGORIES))
width = 0.35

bars1 = ax.bar(x - width/2, f1_euc, width, label='Euclidean', color='#3498db')
bars2 = ax.bar(x + width/2, f1_cos, width, label='Cosine', color='#2ecc71')

ax.set_ylabel('F1-Score')
ax.set_title('F1-Score por Emoción: Euclidean vs Cosine (3 clases)')
ax.set_xticks(x)
ax.set_xticklabels(EMOTION_CATEGORIES)
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"\n✓ Cosine mejora en: {list(comparison_df[comparison_df['Diferencia'] > 0]['Emoción'])}")
print(f"✗ Euclidean mejora en: {list(comparison_df[comparison_df['Diferencia'] < 0]['Emoción'])}")

### Comparación: Raw Pixels + PCA vs LBP

LBP (Local Binary Patterns) es un descriptor de texturas que captura patrones locales.
Comparamos si es mejor que los píxeles crudos proyectados a PCA.

In [ ]:
from skimage.feature import local_binary_pattern

def extract_lbp_features(image, radius=1, n_points=8):
    """Extrae histograma LBP de una imagen"""
    lbp = local_binary_pattern(image, n_points, radius, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), density=True)
    return hist

# Cargar imágenes originales (no PCA) para extraer LBP
# Usamos data/faces/ que tiene los píxeles crudos 48x48

print("Cargando imágenes originales para LBP...")

# Solo las 3 emociones (igual que C++)
# EMOTION_CATEGORIES ya está definido arriba como ['happy', 'neutral', 'sad']

X_raw = []
X_lbp = []
Y_labels = []

for emotion in EMOTION_CATEGORIES:
    filepath = f"../data/faces/{emotion}_faces.csv"
    try:
        faces = pd.read_csv(filepath, header=None).values
        print(f"  {emotion}: {len(faces)} caras")
        
        for face_flat in faces:
            # Reconstruir imagen 48x48
            img = face_flat.reshape(48, 48).astype(np.uint8)
            
            # Raw pixels (normalizado)
            X_raw.append(face_flat / 255.0)
            
            # LBP features
            lbp_hist = extract_lbp_features(img)
            X_lbp.append(lbp_hist)
            
            Y_labels.append(emotion)
    except Exception as e:
        print(f"  {emotion}: Error - {e}")

X_raw = np.array(X_raw)
X_lbp = np.array(X_lbp)
Y_labels = np.array(Y_labels)

print(f"\nDimensiones:")
print(f"  Raw pixels: {X_raw.shape}")
print(f"  LBP: {X_lbp.shape}")

### Balanceo de clases: Submuestreo de Happy

El dataset está desbalanceado:
- happy: 3884 (47%)
- neutral: 2775 (34%)
- sad: 1545 (19%)

Testeamos diferentes niveles de submuestreo de happy para mejorar el balance.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

# Cargar datos frescos del IVF
vectors_all = pd.read_csv("../data/ivf/vectors.csv", header=None).values
labels_raw_all = pd.read_csv("../data/ivf/labels.csv", header=None).values.flatten()

print("Distribución original:")
print(Counter(labels_raw_all))
print(f"  happy(0): {sum(labels_raw_all == 0)}")
print(f"  neutral(1): {sum(labels_raw_all == 1)}")
print(f"  sad(2): {sum(labels_raw_all == 2)}")

# Diferentes ratios de submuestreo para happy
# ratio = n_happy / n_clase_minoritaria (sad)
subsample_ratios = [1.0, 1.5, 2.0, 2.5, 3.0]  # 1.0 = balanceado total

results_balanced = []

for ratio in subsample_ratios:
    n_sad = sum(labels_raw_all == 2)
    n_happy_target = int(n_sad * ratio)
    n_neutral_target = sum(labels_raw_all == 1)  # mantener neutral
    
    # Estrategia de submuestreo
    sampling_strategy = {
        0: min(n_happy_target, sum(labels_raw_all == 0)),  # happy
        1: n_neutral_target,  # neutral (sin cambio)
        2: n_sad  # sad (sin cambio)
    }
    
    rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
    X_resampled, y_resampled = rus.fit_resample(vectors_all, labels_raw_all)
    
    # Labels string
    y_resampled_str = np.array([EMOTION_CATEGORIES[i] for i in y_resampled])
    
    # Split y entrenar
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_resampled, y_resampled_str, test_size=0.2, random_state=42, stratify=y_resampled_str
    )
    
    # KNN k=5 weighted (config actual en C++)
    knn = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='euclidean')
    knn.fit(X_tr, y_tr)
    acc = knn.score(X_te, y_te)
    
    # F1 por clase
    y_pred = knn.predict(X_te)
    prec, rec, f1, _ = precision_recall_fscore_support(y_te, y_pred, labels=EMOTION_CATEGORIES)
    
    results_balanced.append({
        'Ratio happy:sad': f'{ratio:.1f}:1',
        'n_happy': sum(y_resampled == 0),
        'n_neutral': sum(y_resampled == 1),
        'n_sad': sum(y_resampled == 2),
        'Total': len(y_resampled),
        'Accuracy': acc,
        'F1_happy': f1[0],
        'F1_neutral': f1[1],
        'F1_sad': f1[2],
        'F1_macro': np.mean(f1)
    })
    
    print(f"\nRatio {ratio:.1f}:1 - happy:{sum(y_resampled==0)}, neutral:{sum(y_resampled==1)}, sad:{sum(y_resampled==2)}")
    print(f"  Accuracy: {acc:.2%}, F1 macro: {np.mean(f1):.2%}")

results_balanced_df = pd.DataFrame(results_balanced)
display(results_balanced_df)

In [ ]:
# Visualizar resultados del submuestreo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Accuracy y F1 macro por ratio
ax1 = axes[0]
x = range(len(results_balanced_df))
ax1.plot(x, results_balanced_df['Accuracy'], 'o-', label='Accuracy', linewidth=2, markersize=8)
ax1.plot(x, results_balanced_df['F1_macro'], 's-', label='F1 Macro', linewidth=2, markersize=8)
ax1.set_xticks(x)
ax1.set_xticklabels(results_balanced_df['Ratio happy:sad'])
ax1.set_xlabel('Ratio happy:sad')
ax1.set_ylabel('Score')
ax1.set_title('Accuracy y F1 Macro vs Ratio de Submuestreo')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico 2: F1 por clase
ax2 = axes[1]
width = 0.25
x = np.arange(len(results_balanced_df))
ax2.bar(x - width, results_balanced_df['F1_happy'], width, label='happy', color='#2ecc71')
ax2.bar(x, results_balanced_df['F1_neutral'], width, label='neutral', color='#3498db')
ax2.bar(x + width, results_balanced_df['F1_sad'], width, label='sad', color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels(results_balanced_df['Ratio happy:sad'])
ax2.set_xlabel('Ratio happy:sad')
ax2.set_ylabel('F1 Score')
ax2.set_title('F1 Score por Clase vs Ratio')
ax2.legend()
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

# Mejor configuración por F1 macro
best_idx = results_balanced_df['F1_macro'].idxmax()
best_row = results_balanced_df.iloc[best_idx]
print(f"\n>> MEJOR RATIO: {best_row['Ratio happy:sad']}")
print(f"   Accuracy: {best_row['Accuracy']:.2%}")
print(f"   F1 macro: {best_row['F1_macro']:.2%}")
print(f"   F1 happy: {best_row['F1_happy']:.2%}, F1 neutral: {best_row['F1_neutral']:.2%}, F1 sad: {best_row['F1_sad']:.2%}")

In [ ]:
# Matriz de confusión del mejor ratio
best_ratio = float(best_row['Ratio happy:sad'].split(':')[0])

n_sad = sum(labels_raw_all == 2)
n_happy_target = int(n_sad * best_ratio)

sampling_strategy = {
    0: min(n_happy_target, sum(labels_raw_all == 0)),
    1: sum(labels_raw_all == 1),
    2: n_sad
}

rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_best, y_best = rus.fit_resample(vectors_all, labels_raw_all)
y_best_str = np.array([EMOTION_CATEGORIES[i] for i in y_best])

X_tr, X_te, y_tr, y_te = train_test_split(X_best, y_best_str, test_size=0.2, random_state=42, stratify=y_best_str)

knn_best = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='euclidean')
knn_best.fit(X_tr, y_tr)
y_pred_best = knn_best.predict(X_te)

plot_confusion_matrix(y_te, y_pred_best, f"KNN k=5 weighted (ratio {best_row['Ratio happy:sad']})")